# Nested Learning for CAD — bridge chuyển source NN-write v5.1

G3 đã review: giảm44% pipeline time trên cửa sổ16 ảnh, exact gradients/state. **Notebook này chỉ làm G4 bridge**, không thay method hoặc tự chạy hết benchmark.

**Bốn inputs:** source bridge mới `hope_cad_support_v5_1_nn_write_bridge_source_r2.zip` (ZIP hoặc Kaggle giải nén); **original reviewed benchmark NESTED_seed0 paused376/3599** evidence; MVTec AD raw; locked `vit_base_patch8_224.augreg_in21k.safetensors`. Tháo source/output NN-write validation và profiler. Không attach W6/pilot hoặc clone195. Delivery r2 hỗ trợ metadata tar khác thứ tự; không thay phương pháp hoặc bridge budget.

GPU **T4×2**, kernel mới. Giữ PHASE='bridge'; các overrides=None nếu discovery rõ. **Save Version → Save & Run All.** Có heartbeat30 giây. Source verify trước; runtime/backbone/parent checks fail closed. Tar được đọc theo từng arrival, không extract sẵn future normals. Budget60 phút gồm bootstrap và180s export reserve, không phải ETA; không tự tăng budget hoặc retry partial bridge.

Hai owners cùng strict-load active376 và revalidate pending trên T4. Mỗi ảnh376–383 được xử lý ở cả reference/candidate rồi so full state/event trước ảnh kế tiếp. Đủ8 exact mới publish candidate384 cùng new execution lineage; không finalize pending/partial. Không dùng ADtest/GT/VisA.

`bridge_complete_pending_review`: giữ ZIP/log để review trước khi tiếp benchmark. `bridge_incomplete_preserve_parent` hoặc failure: giữ mọi evidence, parent376 vẫn nguyên; không tự resume/rerun hoặc dùng shadow reference. Notebook không expose continuation sau384.

Output `hope_cad_support_v5_1_nn_write_bridge_evidence.zip` và log lưu tại `D:/src/DATN/.agents/artifacts/runs/support_v5_1_nn_write_bridge_01/session_01/`. Đọc `docs/main_spec/migration_nn_write_v5_1.md` và `migration_nn_write_v5_1_implementation.md` trước chạy.


In [ ]:
PHASE='bridge'
SOURCE_OVERRIDE=None
PARENT_OVERRIDE=None  # exact reviewed benchmark376 ZIP or extracted evidence root
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None

from pathlib import Path,PurePosixPath
import hashlib,json,zipfile,time,os,sys,shutil
STARTED=time.monotonic()
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
sys.dont_write_bytecode=True
INPUT=Path('/kaggle/input');ROOT=Path('/tmp/hope_nn_write_bridge_source_r2')
WORK=Path('/kaggle/working/hope_nn_write_bridge')
ASSETS=Path('/tmp/hope_nn_write_bridge_assets')
OUTPUT=Path('/kaggle/working/hope_cad_support_v5_1_nn_write_bridge_evidence.zip')
PINNED_CONTRACT='691d70257e154c70e4e9f3bc33a03c665b51dc5f14c4e4f6cd837215e093f427'
print('[BRIDGE BOOTSTRAP] locating source; elapsed',round(time.monotonic()-STARTED,1),'s',flush=True)
def safe(root,name):
    p=PurePosixPath(name)
    if p.is_absolute() or p.as_posix()!=name or '..' in p.parts or '\\' in name or ':' in name:raise ValueError('Unsafe source path')
    out=root/name
    if not out.resolve().is_relative_to(root.resolve()):raise ValueError('Escaping source path')
    return out
def manifest(p):
    if p.is_dir():return json.loads((p/'bridge_manifest.json').read_text())
    with zipfile.ZipFile(p) as z:return json.loads(z.read('bridge_manifest.json'))
choices=[Path(SOURCE_OVERRIDE)] if SOURCE_OVERRIDE else [p.parent for p in INPUT.rglob('bridge_manifest.json')]+list(INPUT.rglob('*.zip'))
valid=[]
for p in choices:
    try:
        m=manifest(p)
        if m.get('schema')=='hope_support_v5_1_nn_write_bridge_source_v1':valid.append((p,m))
    except (OSError,KeyError,ValueError,zipfile.BadZipFile):
        if SOURCE_OVERRIDE:raise
identities={json.dumps(m,sort_keys=True) for p,m in valid}
if len(identities)!=1:raise ValueError('Expected ONE bridge source identity; set SOURCE_OVERRIDE')
source,m=sorted(valid,key=lambda v:str(v[0]))[0]
if m['contract_sha256']!=PINNED_CONTRACT:raise ValueError('Notebook/source contract version mismatch')
ROOT.mkdir(parents=True,exist_ok=True)
z=None if source.is_dir() else zipfile.ZipFile(source)
try:
    if z and (len(z.namelist())!=len(set(z.namelist())) or set(z.namelist())!=set(m['files'])|{'bridge_manifest.json'}):raise ValueError('Source ZIP inventory mismatch')
    for name,expected in m['files'].items():
        original=safe(source,name) if source.is_dir() else None
        if original is not None and original.is_symlink():raise ValueError('Source symlink')
        raw=original.read_bytes() if original is not None else z.read(name)
        if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Source payload SHA mismatch: '+name)
        dest=safe(ROOT,name);dest.parent.mkdir(parents=True,exist_ok=True)
        if dest.exists() and dest.read_bytes()!=raw:raise ValueError('Existing source differs; use a fresh kernel/session')
        dest.write_bytes(raw)
    target=ROOT/'bridge_manifest.json';raw=json.dumps(m,sort_keys=True).encode()
    if target.exists() and json.loads(target.read_bytes())!=m:raise ValueError('Existing delivery differs')
    target.write_bytes(raw)
finally:
    if z:z.close()
for name in ('training','models','dataset','notebook_support'):
    module=sys.modules.get(name)
    if module:
        paths=list(getattr(module,'__path__',[]))
        if not paths or any(not Path(p).resolve().is_relative_to(ROOT) for p in paths):raise ValueError('Mixed source imports; restart kernel')
sys.path.insert(0,str(ROOT));os.chdir(ROOT)
print('[BRIDGE BOOTSTRAP] verified',len(m['files']),'files;',m['origin'],flush=True)
print('Fixed60min budget;8newarrivals376-383 lockstep; export384 only after exact; no auto benchmark continuation',flush=True)


In [ ]:
from notebook_support.support_migration_v5_1.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,inputs=str(INPUT),work=str(WORK),assets=str(ASSETS),output=str(OUTPUT),started=STARTED,
    PARENT_OVERRIDE=PARENT_OVERRIDE,MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))
